# Stone-Level Detection Evaluation (Bidirectional Approach)

This notebook evaluates segmentation performance at the **individual stone level**, measuring both **coverage** (how well each stone is detected) and **separation** (whether stones are kept distinct vs. merged).

## Key Concepts:

**Coverage (Recall)**: What fraction of a GT stone's pixels are covered by same-class predictions?
- Multiple same-class predictions covering one GT stone are **summed up** (fragmentation is OK)
- Only predictions matching the GT stone's class count toward coverage

**Separation**: Are the predictions "dedicated" to this stone, or do they extend into neighboring stones?
- High separation = predictions mostly stay within this stone's boundaries
- Low separation = predictions extend far beyond this stone (merged with neighbors)

**Detection Status**:
- `detected`: Good coverage AND good separation
- `merged`: Good coverage BUT poor separation (stone is part of a larger blob)
- `undetected`: Poor coverage (stone was missed)

## Workflow:
1. **Set parameters** (paths, thresholds)
2. **Extract GT stones** using connected components
3. **Evaluate each GT stone** against predictions (coverage + separation)
4. **Visualize and compare** across models

---
## Cell 1: Parameters

Configure paths and evaluation thresholds here.

In [ ]:
# ============================================================
# FILE PATHS - EDIT THESE
# ============================================================

# Ground truth mask path
GT_MASK_PATH = r"C:\Users\admin\Desktop\OrdnerAmGArtikel2025\AWS_TRAINING\2025-08-10_4classEX\testing\02_masks\H_Bf_5_png-ortho.png"

# Prediction paths for all three models
PRED_PATHS = {
    '3-channel': r"C:\Users\admin\Desktop\OrdnerAmGArtikel2025\AWS_TRAINING\2025-08-10_4classEX\testing\05_outputs\3-channel\H_Bf_5_png-DEM_normalmap_RAW_combined_3ch.png",   # Geometry-only (normal maps)
    '4-channel': r"C:\Users\admin\Desktop\OrdnerAmGArtikel2025\AWS_TRAINING\2025-08-10_4classEX\testing\05_outputs\4-channel\H_Bf_5_png-ortho_RAW_combined_4ch.png",   # Appearance-only (RGB + alpha)
    '7-channel': r"C:\Users\admin\Desktop\OrdnerAmGArtikel2025\AWS_TRAINING\2025-08-10_4classEX\testing\05_outputs\7-channel\H_Bf_5_png-ortho_RAW_combined.png"    # Combined (RGB + alpha + normals)
}

# Output directory for results
OUTPUT_DIR = r"C:\Users\admin\Desktop\OrdnerAmGArtikel2025\AWS_TRAINING\2025-08-10_4classEX\testing\09_Stone_Detection\Wall1_90_90"

# ============================================================
# EVALUATION THRESHOLDS - ADJUST THESE TO TUNE SENSITIVITY
# ============================================================

# COVERAGE THRESHOLD (Recall)
# ---------------------------
# What fraction of a GT stone's pixels must be covered by same-class predictions
# for the stone to be considered "detected"?
#
# - 0.3 = lenient (30% coverage is enough)
# - 0.5 = moderate (half the stone must be covered)
# - 0.7 = strict (most of the stone must be covered)
#
# Note: Multiple predictions of the same class are summed up, so fragmented
# predictions (3 small blobs covering one stone) can still pass this threshold.
COVERAGE_THRESHOLD = 0.9

# SEPARATION THRESHOLD
# --------------------
# How "dedicated" must predictions be to this specific stone?
# Measures whether predictions stay within the stone's boundaries or extend
# into neighboring stones (indicating a merge).
#
# - 0.3 = lenient (tolerates significant merging/bridging)
# - 0.5 = moderate (allows minor bridging, penalizes major merges)
# - 0.7 = strict (predictions must mostly stay within this stone)
#
# Lower this value if you want to tolerate more "bridging" between stones.
# Raise it if even minor bridging should count as a merge.
SEPARATION_THRESHOLD = 0.9

# MINIMUM STONE SIZE
# ------------------
# Ignore GT stones smaller than this (filters noise/artifacts)
MIN_STONE_SIZE = 100  # pixels

# ============================================================
# CLASS CONFIGURATION
# ============================================================

CLASS_NAMES = ['Background', 'Ashlar', 'Polygonal', 'Quarry Stone']

MODEL_COLORS = {
    '3-channel': '#e74c3c',  # Red
    '4-channel': '#3498db',  # Blue
    '7-channel': '#2ecc71'   # Green
}

# ============================================================
# PRINT CONFIGURATION
# ============================================================

print("CONFIGURATION")
print("="*60)
print(f"\nFile Paths:")
print(f"  Ground Truth: {GT_MASK_PATH}")
for model_name, path in PRED_PATHS.items():
    print(f"  {model_name}: {path}")
print(f"  Output: {OUTPUT_DIR}")
print(f"\nThresholds:")
print(f"  Coverage (Recall): {COVERAGE_THRESHOLD}")
print(f"    → Stone needs ≥{COVERAGE_THRESHOLD*100:.0f}% of pixels covered by same-class predictions")
print(f"  Separation: {SEPARATION_THRESHOLD}")
print(f"    → Predictions must be ≥{SEPARATION_THRESHOLD*100:.0f}% 'dedicated' to this stone")
print(f"  Min Stone Size: {MIN_STONE_SIZE} pixels")
print("\n" + "="*60)



---
## Cell 2: Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from PIL import Image
import pandas as pd
from scipy import ndimage
from skimage.measure import regionprops
from typing import Dict, Tuple, List, Optional
from dataclasses import dataclass
import os
import warnings
warnings.filterwarnings('ignore')

Image.MAX_IMAGE_PIXELS = None

# RGB to class mapping
RGB_TO_CLASS = {
    (0, 0, 0): 0,       # Black -> Background
    (0, 0, 255): 1,     # Blue -> Ashlar
    (255, 0, 0): 2,     # Red -> Polygonal
    (255, 255, 0): 3    # Yellow -> Quarry Stone
}

CLASS_COLORS_RGB = {
    0: [0, 0, 0],       # Background - Black
    1: [0, 0, 255],     # Ashlar - Blue
    2: [255, 0, 0],     # Polygonal - Red
    3: [255, 255, 0]    # Quarry Stone - Yellow
}

CLASS_COLORS_HEX = ['#000000', '#0000FF', '#FF0000', '#FFFF00']

print("✓ Imports complete.")

---
## Cell 3: Data Structures

In [ ]:
@dataclass
class Stone:
    """Represents a single GT stone."""
    stone_id: int
    class_id: int
    class_name: str
    pixel_count: int
    mask: np.ndarray
    centroid: Tuple[float, float]
    bbox: Tuple[int, int, int, int]


@dataclass 
class StoneEvaluation:
    """Evaluation result for a single GT stone."""
    gt_stone: Stone
    
    # Coverage metrics
    coverage: float              # Fraction of GT covered by same-class predictions (0-1)
    wrong_class_coverage: float  # Fraction covered by wrong-class predictions (0-1)
    uncovered: float             # Fraction not covered by any prediction (0-1)
    
    # Separation metrics
    separation: float            # How dedicated are predictions to this stone (0-1)
    
    # Final status
    status: str                  # 'detected', 'merged', 'undetected'
    
    # Additional info
    num_predictions_involved: int  # How many prediction blobs touch this stone


print("✓ Data structures defined.")

---
## Cell 4: Helper Functions

In [ ]:
def rgb_to_class_mask(rgb_image: np.ndarray, verbose: bool = True) -> np.ndarray:
    """Convert RGB mask to class indices."""
    height, width = rgb_image.shape[:2]
    class_mask = np.zeros((height, width), dtype=np.uint8)
    
    for rgb_tuple, class_idx in RGB_TO_CLASS.items():
        color_mask = np.all(rgb_image == rgb_tuple, axis=2)
        class_mask[color_mask] = class_idx
    
    # Handle unmapped pixels
    mapped_pixels = np.zeros((height, width), dtype=bool)
    for rgb_tuple in RGB_TO_CLASS.keys():
        mapped_pixels |= np.all(rgb_image == rgb_tuple, axis=2)
    
    unmapped_count = np.sum(~mapped_pixels)
    if unmapped_count > 0:
        if verbose:
            print(f"    Note: {unmapped_count} unmapped pixels -> nearest class")
        unmapped_indices = np.where(~mapped_pixels)
        for i in range(len(unmapped_indices[0])):
            y, x = unmapped_indices[0][i], unmapped_indices[1][i]
            pixel_rgb = rgb_image[y, x]
            min_dist = float('inf')
            nearest_class = 0
            for rgb_tuple, class_idx in RGB_TO_CLASS.items():
                dist = np.sqrt(np.sum((pixel_rgb.astype(float) - np.array(rgb_tuple))**2))
                if dist < min_dist:
                    min_dist = dist
                    nearest_class = class_idx
            class_mask[y, x] = nearest_class
    
    return class_mask


def extract_stones(class_mask: np.ndarray, class_names: List[str], min_size: int) -> List[Stone]:
    """Extract individual stones from GT mask using connected components."""
    stone_binary = (class_mask > 0).astype(np.uint8)
    labeled_array, _ = ndimage.label(stone_binary)
    
    stones = []
    for region in regionprops(labeled_array):
        if region.area < min_size:
            continue
        
        stone_mask = (labeled_array == region.label)
        stone_classes = class_mask[stone_mask]
        class_id = int(np.bincount(stone_classes).argmax())
        
        stones.append(Stone(
            stone_id=region.label,
            class_id=class_id,
            class_name=class_names[class_id],
            pixel_count=region.area,
            mask=stone_mask,
            centroid=(region.centroid[0], region.centroid[1]),
            bbox=(region.bbox[0], region.bbox[1], region.bbox[2], region.bbox[3])
        ))
    
    return stones


def extract_prediction_regions(pred_mask: np.ndarray, min_size: int = 10) -> Tuple[np.ndarray, Dict]:
    """
    Extract labeled prediction regions (connected components) from prediction mask.
    
    Returns:
        labeled_preds: Array where each prediction region has a unique ID
        pred_info: Dict mapping pred_id -> {'class_id': int, 'pixel_count': int}
    """
    pred_binary = (pred_mask > 0).astype(np.uint8)
    labeled_preds, num_preds = ndimage.label(pred_binary)
    
    pred_info = {}
    for region in regionprops(labeled_preds):
        if region.area < min_size:
            # Zero out small regions
            labeled_preds[labeled_preds == region.label] = 0
            continue
        
        region_mask = (labeled_preds == region.label)
        region_classes = pred_mask[region_mask]
        class_id = int(np.bincount(region_classes[region_classes > 0]).argmax()) if np.any(region_classes > 0) else 0
        
        pred_info[region.label] = {
            'class_id': class_id,
            'pixel_count': region.area
        }
    
    return labeled_preds, pred_info


print("✓ Helper functions defined.")

---
## Cell 5: Core Evaluation Logic

This is where the bidirectional evaluation happens:
- **Coverage**: Sum up all same-class prediction pixels within the GT stone
- **Separation**: Measure how "dedicated" each prediction is to this stone

In [ ]:
def evaluate_stone(
    gt_stone: Stone,
    pred_mask: np.ndarray,
    labeled_preds: np.ndarray,
    pred_info: Dict,
    coverage_threshold: float,
    separation_threshold: float
) -> StoneEvaluation:
    """
    Evaluate a single GT stone against predictions.
    
    Coverage Calculation:
    ---------------------
    For the GT stone's pixels, count how many are:
    - Covered by same-class predictions (correct)
    - Covered by different-class predictions (misclassified)
    - Not covered by any stone prediction (missed)
    
    Multiple same-class predictions are SUMMED UP - this handles fragmentation.
    
    Separation Calculation:
    -----------------------
    For each prediction region overlapping this GT stone:
    - Calculate "commitment" = (overlap with this GT) / (total prediction size)
    - High commitment = prediction is dedicated to this stone
    - Low commitment = prediction extends far beyond (into other stones)
    
    Final separation = weighted average of commitments, weighted by overlap size.
    This means predictions that contribute more to this stone have more influence.
    """
    gt_class = gt_stone.class_id
    gt_pixels = gt_stone.pixel_count
    
    # Get prediction values within this GT stone's area
    pred_within_stone = pred_mask[gt_stone.mask]
    pred_labels_within = labeled_preds[gt_stone.mask]
    
    # =========================================
    # COVERAGE CALCULATION
    # =========================================
    
    # Count pixels by prediction class
    same_class_pixels = np.sum(pred_within_stone == gt_class)
    wrong_class_pixels = np.sum((pred_within_stone > 0) & (pred_within_stone != gt_class))
    uncovered_pixels = np.sum(pred_within_stone == 0)
    
    coverage = same_class_pixels / gt_pixels
    wrong_class_coverage = wrong_class_pixels / gt_pixels
    uncovered = uncovered_pixels / gt_pixels
    
    # =========================================
    # SEPARATION CALCULATION
    # =========================================
    
    # Find all prediction regions that overlap with this GT stone
    unique_pred_labels = np.unique(pred_labels_within)
    unique_pred_labels = unique_pred_labels[unique_pred_labels > 0]  # Exclude background
    
    # Only consider same-class predictions for separation
    same_class_pred_labels = [
        lbl for lbl in unique_pred_labels 
        if lbl in pred_info and pred_info[lbl]['class_id'] == gt_class
    ]
    
    if len(same_class_pred_labels) == 0:
        # No same-class predictions overlap with this stone
        separation = 0.0
        num_preds = 0
    else:
        # Calculate weighted average commitment
        total_weight = 0
        weighted_commitment_sum = 0
        
        for pred_label in same_class_pred_labels:
            # How many pixels of this prediction are within the GT stone?
            overlap = np.sum(pred_labels_within == pred_label)
            
            # Total size of this prediction region
            pred_total_size = pred_info[pred_label]['pixel_count']
            
            # Commitment: what fraction of the prediction is within this GT stone?
            commitment = overlap / pred_total_size
            
            # Weight by overlap (predictions that contribute more have more influence)
            weight = overlap
            
            total_weight += weight
            weighted_commitment_sum += weight * commitment
        
        separation = weighted_commitment_sum / total_weight if total_weight > 0 else 0.0
        num_preds = len(same_class_pred_labels)
    
    # =========================================
    # DETERMINE STATUS
    # =========================================
    
    if coverage >= coverage_threshold:
        if separation >= separation_threshold:
            status = 'detected'   # Good coverage AND good separation
        else:
            status = 'merged'     # Good coverage BUT poor separation (part of larger blob)
    else:
        status = 'undetected'     # Poor coverage
    
    return StoneEvaluation(
        gt_stone=gt_stone,
        coverage=coverage,
        wrong_class_coverage=wrong_class_coverage,
        uncovered=uncovered,
        separation=separation,
        status=status,
        num_predictions_involved=num_preds
    )


def evaluate_all_stones(
    gt_stones: List[Stone],
    pred_mask: np.ndarray,
    coverage_threshold: float,
    separation_threshold: float
) -> List[StoneEvaluation]:
    """Evaluate all GT stones against predictions."""
    
    # Extract prediction regions once
    labeled_preds, pred_info = extract_prediction_regions(pred_mask)
    
    evaluations = []
    for gt_stone in gt_stones:
        eval_result = evaluate_stone(
            gt_stone, pred_mask, labeled_preds, pred_info,
            coverage_threshold, separation_threshold
        )
        evaluations.append(eval_result)
    
    return evaluations


print("✓ Core evaluation logic defined.")

---
## Cell 6: Metrics Calculation

In [ ]:
def calculate_metrics(evaluations: List[StoneEvaluation], class_names: List[str]) -> Dict:
    """Calculate summary metrics from stone evaluations."""
    
    total = len(evaluations)
    detected = sum(1 for e in evaluations if e.status == 'detected')
    merged = sum(1 for e in evaluations if e.status == 'merged')
    undetected = sum(1 for e in evaluations if e.status == 'undetected')
    
    # Mean scores
    mean_coverage = np.mean([e.coverage for e in evaluations])
    mean_separation = np.mean([e.separation for e in evaluations if e.coverage > 0])  # Only for covered stones
    mean_wrong_class = np.mean([e.wrong_class_coverage for e in evaluations])
    
    # Per-class breakdown
    per_class = {}
    for class_idx, class_name in enumerate(class_names):
        if class_idx == 0:  # Skip background
            continue
        
        class_evals = [e for e in evaluations if e.gt_stone.class_id == class_idx]
        if len(class_evals) == 0:
            continue
            
        class_total = len(class_evals)
        class_detected = sum(1 for e in class_evals if e.status == 'detected')
        class_merged = sum(1 for e in class_evals if e.status == 'merged')
        class_undetected = sum(1 for e in class_evals if e.status == 'undetected')
        
        per_class[class_name] = {
            'total': class_total,
            'detected': class_detected,
            'merged': class_merged,
            'undetected': class_undetected,
            'detection_rate': class_detected / class_total,
            'merge_rate': class_merged / class_total,
            'mean_coverage': np.mean([e.coverage for e in class_evals]),
            'mean_separation': np.mean([e.separation for e in class_evals if e.coverage > 0]) if any(e.coverage > 0 for e in class_evals) else 0
        }
    
    return {
        'total_stones': total,
        'detected': detected,
        'merged': merged,
        'undetected': undetected,
        'detection_rate': detected / total if total > 0 else 0,
        'merge_rate': merged / total if total > 0 else 0,
        'mean_coverage': mean_coverage,
        'mean_separation': mean_separation if not np.isnan(mean_separation) else 0,
        'mean_wrong_class_coverage': mean_wrong_class,
        'per_class': per_class
    }


print("✓ Metrics calculation defined.")

---
## Cell 7: Load Ground Truth

In [ ]:
print("Loading Ground Truth...")
print("="*60)

gt_rgb = np.array(Image.open(GT_MASK_PATH).convert('RGB'))
gt_mask = rgb_to_class_mask(gt_rgb)
print(f"  Image shape: {gt_mask.shape}")

gt_stones = extract_stones(gt_mask, CLASS_NAMES, MIN_STONE_SIZE)
print(f"  Total stones extracted: {len(gt_stones)}")

print(f"\n  Per-class counts:")
for class_idx, class_name in enumerate(CLASS_NAMES):
    if class_idx == 0:
        continue
    count = sum(1 for s in gt_stones if s.class_id == class_idx)
    print(f"    {class_name}: {count}")

print("\n" + "="*60)
print("✓ Ground truth loaded.")

---
## Cell 8: Visualize GT Stones

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Class mask
cmap = ListedColormap(CLASS_COLORS_HEX)
axes[0].imshow(gt_mask, cmap=cmap, vmin=0, vmax=3)
axes[0].set_title(f'Ground Truth Classes\n({len(gt_stones)} stones)', fontsize=12, fontweight='bold')
axes[0].axis('off')

# Instance map (each stone different color)
instance_map = np.zeros(gt_mask.shape, dtype=np.int32)
for stone in gt_stones:
    instance_map[stone.mask] = stone.stone_id

np.random.seed(42)
n_colors = max(instance_map.max() + 1, len(gt_stones) + 1)
random_colors = np.random.rand(n_colors, 3)
random_colors[0] = [0, 0, 0]
instance_cmap = ListedColormap(random_colors)

axes[1].imshow(instance_map, cmap=instance_cmap)
axes[1].set_title('Individual Stones (Instance Map)', fontsize=12, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, '01_ground_truth_instances.png'), dpi=200, bbox_inches='tight')
plt.show()

---
## Cell 9: Load Predictions

In [ ]:
pred_masks = {}

print("Loading Predictions...")
print("="*60)

for model_name, pred_path in PRED_PATHS.items():
    print(f"\n  {model_name}:")
    pred_rgb = np.array(Image.open(pred_path).convert('RGB'))
    pred_mask = rgb_to_class_mask(pred_rgb, verbose=True)
    
    assert pred_mask.shape == gt_mask.shape, f"Shape mismatch!"
    pred_masks[model_name] = pred_mask
    print(f"    Shape: {pred_mask.shape} ✓")

print("\n" + "="*60)
print("✓ All predictions loaded.")

---
## Cell 10: Run Evaluation

In [ ]:
all_evaluations = {}
all_metrics = {}

print("Running Stone-Level Evaluation...")
print(f"  Coverage threshold: {COVERAGE_THRESHOLD}")
print(f"  Separation threshold: {SEPARATION_THRESHOLD}")
print("="*80)

for model_name, pred_mask in pred_masks.items():
    print(f"\n{model_name.upper()}")
    print("-"*50)
    
    # Evaluate all stones
    evaluations = evaluate_all_stones(
        gt_stones, pred_mask, 
        COVERAGE_THRESHOLD, SEPARATION_THRESHOLD
    )
    all_evaluations[model_name] = evaluations
    
    # Calculate metrics
    metrics = calculate_metrics(evaluations, CLASS_NAMES)
    all_metrics[model_name] = metrics
    
    # Print summary
    print(f"  Total GT stones: {metrics['total_stones']}")
    print(f"")
    print(f"  Detected:   {metrics['detected']:3d} ({metrics['detection_rate']*100:5.1f}%)")
    print(f"  Merged:     {metrics['merged']:3d} ({metrics['merge_rate']*100:5.1f}%)")
    print(f"  Undetected: {metrics['undetected']:3d} ({(1-metrics['detection_rate']-metrics['merge_rate'])*100:5.1f}%)")
    print(f"")
    print(f"  Mean Coverage:   {metrics['mean_coverage']*100:5.1f}%")
    print(f"  Mean Separation: {metrics['mean_separation']*100:5.1f}%")
    
    if metrics['mean_wrong_class_coverage'] > 0.01:
        print(f"  Mean Wrong-Class: {metrics['mean_wrong_class_coverage']*100:5.1f}%")
    
    print(f"\n  Per-class breakdown:")
    for class_name, class_metrics in metrics['per_class'].items():
        print(f"    {class_name:15s}: {class_metrics['detected']:2d}/{class_metrics['total']:2d} detected "
              f"({class_metrics['detection_rate']*100:4.1f}%), "
              f"{class_metrics['merged']:2d} merged")

print("\n" + "="*80)
print("✓ Evaluation complete.")

---
## Cell 11: Visualize Coverage vs Separation (Scatter Plot)

This plot shows each GT stone as a point, with:
- X-axis: Coverage (how much of the stone is covered by same-class predictions)
- Y-axis: Separation (how dedicated predictions are to this stone)

The thresholds divide the plot into quadrants:
- Top-right: Detected (good coverage + good separation)
- Top-left: Undetected (poor coverage)
- Bottom-right: Merged (good coverage + poor separation)
- Bottom-left: Undetected (poor coverage)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, model_name in enumerate(PRED_PATHS.keys()):
    ax = axes[idx]
    evaluations = all_evaluations[model_name]
    
    # Scatter plot
    for eval_result in evaluations:
        color = {
            'detected': '#2ecc71',
            'merged': '#f39c12', 
            'undetected': '#e74c3c'
        }[eval_result.status]
        
        ax.scatter(eval_result.coverage, eval_result.separation, 
                   c=color, s=50, alpha=0.6, edgecolors='black', linewidth=0.5)
    
    # Draw threshold lines
    ax.axvline(COVERAGE_THRESHOLD, color='gray', linestyle='--', linewidth=2, alpha=0.7)
    ax.axhline(SEPARATION_THRESHOLD, color='gray', linestyle='--', linewidth=2, alpha=0.7)
    
    # Labels for quadrants
    ax.text(0.75, 0.85, 'DETECTED', transform=ax.transAxes, fontsize=10, 
            fontweight='bold', color='#2ecc71', ha='center')
    ax.text(0.75, 0.15, 'MERGED', transform=ax.transAxes, fontsize=10,
            fontweight='bold', color='#f39c12', ha='center')
    ax.text(0.25, 0.5, 'UNDETECTED', transform=ax.transAxes, fontsize=10,
            fontweight='bold', color='#e74c3c', ha='center')
    
    ax.set_xlabel('Coverage (Recall)', fontsize=11)
    ax.set_ylabel('Separation', fontsize=11)
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.set_title(f'{model_name}\nDetected: {all_metrics[model_name]["detection_rate"]*100:.1f}%', 
                 fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3)

# Legend
legend_elements = [
    Patch(facecolor='#2ecc71', label='Detected'),
    Patch(facecolor='#f39c12', label='Merged'),
    Patch(facecolor='#e74c3c', label='Undetected')
]
fig.legend(handles=legend_elements, loc='upper center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, 1.02))

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, '02_coverage_vs_separation_scatter.png'), dpi=200, bbox_inches='tight')
plt.show()

---
## Cell 12: Detection Status Breakdown (Bar Charts)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_names = list(PRED_PATHS.keys())
x = np.arange(len(model_names))
width = 0.6

# Stacked bar chart
detected_counts = [all_metrics[m]['detected'] for m in model_names]
merged_counts = [all_metrics[m]['merged'] for m in model_names]
undetected_counts = [all_metrics[m]['undetected'] for m in model_names]

axes[0].bar(x, detected_counts, width, label='Detected', color='#2ecc71', edgecolor='black')
axes[0].bar(x, merged_counts, width, bottom=detected_counts, label='Merged', color='#f39c12', edgecolor='black')
axes[0].bar(x, undetected_counts, width, 
            bottom=[d+m for d,m in zip(detected_counts, merged_counts)],
            label='Undetected', color='#e74c3c', edgecolor='black')

# Add count labels
for i, (d, m, u) in enumerate(zip(detected_counts, merged_counts, undetected_counts)):
    if d > 0:
        axes[0].text(i, d/2, f'{d}', ha='center', va='center', fontweight='bold', color='white', fontsize=11)
    if m > 0:
        axes[0].text(i, d + m/2, f'{m}', ha='center', va='center', fontweight='bold', fontsize=11)
    if u > 0:
        axes[0].text(i, d + m + u/2, f'{u}', ha='center', va='center', fontweight='bold', color='white', fontsize=11)

axes[0].set_ylabel('Number of Stones', fontsize=12)
axes[0].set_title('Detection Status Breakdown', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_names, fontsize=11)
axes[0].legend(loc='upper right', fontsize=10)
axes[0].grid(axis='y', alpha=0.3)

# Mean Coverage vs Mean Separation
coverages = [all_metrics[m]['mean_coverage'] * 100 for m in model_names]
separations = [all_metrics[m]['mean_separation'] * 100 for m in model_names]

x2 = np.arange(len(model_names))
width2 = 0.35

bars1 = axes[1].bar(x2 - width2/2, coverages, width2, label='Mean Coverage', color='#3498db', edgecolor='black')
bars2 = axes[1].bar(x2 + width2/2, separations, width2, label='Mean Separation', color='#9b59b6', edgecolor='black')

for bar, val in zip(bars1, coverages):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
for bar, val in zip(bars2, separations):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

axes[1].set_ylabel('Score (%)', fontsize=12)
axes[1].set_title('Mean Coverage & Separation', fontsize=14, fontweight='bold')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(model_names, fontsize=11)
axes[1].legend(loc='upper right', fontsize=10)
axes[1].set_ylim(0, 110)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, '03_detection_breakdown_bars.png'), dpi=200, bbox_inches='tight')
plt.show()

---
## Cell 13: Spatial Visualization of Detection Results

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 16))

# Ground truth
cmap = ListedColormap(CLASS_COLORS_HEX)
axes[0, 0].imshow(gt_mask, cmap=cmap, vmin=0, vmax=3)
axes[0, 0].set_title(f'Ground Truth\n({len(gt_stones)} stones)', fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

# Detection results for each model
positions = [(0, 1), (1, 0), (1, 1)]
model_list = list(PRED_PATHS.keys())

for (row, col), model_name in zip(positions, model_list):
    evaluations = all_evaluations[model_name]
    metrics = all_metrics[model_name]
    
    # Create visualization
    vis_img = np.zeros((*gt_mask.shape, 3), dtype=np.uint8)
    vis_img[gt_mask == 0] = [40, 40, 40]  # Dark gray background
    
    for eval_result in evaluations:
        if eval_result.status == 'detected':
            color = [46, 204, 113]   # Green
        elif eval_result.status == 'merged':
            color = [243, 156, 18]   # Orange
        else:
            color = [231, 76, 60]    # Red
        
        vis_img[eval_result.gt_stone.mask] = color
    
    axes[row, col].imshow(vis_img)
    axes[row, col].set_title(
        f'{model_name}\n'
        f'Detected: {metrics["detection_rate"]*100:.1f}% | '
        f'Merged: {metrics["merge_rate"]*100:.1f}%',
        fontsize=11, fontweight='bold'
    )
    axes[row, col].axis('off')

# Legend
legend_elements = [
    Patch(facecolor='#2ecc71', label='Detected (good coverage + separation)'),
    Patch(facecolor='#f39c12', label='Merged (good coverage, poor separation)'),
    Patch(facecolor='#e74c3c', label='Undetected (poor coverage)')
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, 0.02))

plt.tight_layout(rect=[0, 0.05, 1, 1])
fig.savefig(os.path.join(OUTPUT_DIR, '04_detection_status_map.png'), dpi=200, bbox_inches='tight')
plt.show()

---
## Cell 14: Coverage Heatmap Visualization

Shows per-stone coverage as a heatmap (darker = better coverage)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

for idx, model_name in enumerate(PRED_PATHS.keys()):
    evaluations = all_evaluations[model_name]
    
    # Create coverage heatmap
    coverage_map = np.zeros(gt_mask.shape, dtype=np.float32)
    for eval_result in evaluations:
        coverage_map[eval_result.gt_stone.mask] = eval_result.coverage
    
    # Mask background
    coverage_map_masked = np.ma.masked_where(gt_mask == 0, coverage_map)
    
    im = axes[idx].imshow(coverage_map_masked, cmap='RdYlGn', vmin=0, vmax=1)
    axes[idx].set_title(f'{model_name}\nPer-Stone Coverage', fontsize=12, fontweight='bold')
    axes[idx].axis('off')

# Adjust layout first to reserve space at bottom, then add colorbar
plt.tight_layout(rect=[0, 0.1, 1, 1])
cbar = fig.colorbar(im, ax=axes, orientation='horizontal', fraction=0.04, pad=0.08, aspect=40)
cbar.set_label('Coverage (0 = none, 1 = fully covered)', fontsize=11)

fig.savefig(os.path.join(OUTPUT_DIR, '05_coverage_heatmap.png'), dpi=200, bbox_inches='tight')
plt.show()

---
## Cell 15: Separation Heatmap Visualization

Shows per-stone separation as a heatmap (darker = better separation, less merging)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

for idx, model_name in enumerate(PRED_PATHS.keys()):
    evaluations = all_evaluations[model_name]
    
    # Create separation heatmap
    separation_map = np.zeros(gt_mask.shape, dtype=np.float32)
    for eval_result in evaluations:
        separation_map[eval_result.gt_stone.mask] = eval_result.separation
    
    # Mask background
    separation_map_masked = np.ma.masked_where(gt_mask == 0, separation_map)
    
    im = axes[idx].imshow(separation_map_masked, cmap='RdYlGn', vmin=0, vmax=1)
    axes[idx].set_title(f'{model_name}\nPer-Stone Separation', fontsize=12, fontweight='bold')
    axes[idx].axis('off')

# Adjust layout first to reserve space at bottom, then add colorbar
plt.tight_layout(rect=[0, 0.1, 1, 1])
cbar = fig.colorbar(im, ax=axes, orientation='horizontal', fraction=0.04, pad=0.08, aspect=40)
cbar.set_label('Separation (0 = fully merged, 1 = perfectly separated)', fontsize=11)

fig.savefig(os.path.join(OUTPUT_DIR, '06_separation_heatmap.png'), dpi=200, bbox_inches='tight')
plt.show()

---
## Cell 16: Comparative Summary Table

In [ ]:
# Build summary table
summary_data = []

for model_name in PRED_PATHS.keys():
    metrics = all_metrics[model_name]
    row = {
        'Model': model_name,
        'Total': metrics['total_stones'],
        'Detected': metrics['detected'],
        'Merged': metrics['merged'],
        'Undetected': metrics['undetected'],
        'Detection_Rate': metrics['detection_rate'],
        'Merge_Rate': metrics['merge_rate'],
        'Mean_Coverage': metrics['mean_coverage'],
        'Mean_Separation': metrics['mean_separation']
    }
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)

# Display
print("\n" + "="*100)
print(f"STONE-LEVEL DETECTION SUMMARY")
print(f"Coverage threshold: {COVERAGE_THRESHOLD} | Separation threshold: {SEPARATION_THRESHOLD}")
print("="*100)

display_df = summary_df.copy()
for col in ['Detection_Rate', 'Merge_Rate', 'Mean_Coverage', 'Mean_Separation']:
    display_df[col] = display_df[col].apply(lambda x: f'{x*100:.1f}%')

print(display_df.to_string(index=False))
print("="*100)

# Highlight best
best_det = summary_df.loc[summary_df['Detection_Rate'].idxmax(), 'Model']
best_sep = summary_df.loc[summary_df['Mean_Separation'].idxmax(), 'Model']
print(f"\n→ Best detection rate: {best_det} ({summary_df['Detection_Rate'].max()*100:.1f}%)")
print(f"→ Best separation: {best_sep} ({summary_df['Mean_Separation'].max()*100:.1f}%)")

---
## Cell 17: Threshold Sensitivity Analysis

How do detection rates change with different threshold settings?

In [ ]:
# Test different threshold combinations
coverage_thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
separation_thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Effect of coverage threshold (fixed separation)
for model_name in PRED_PATHS.keys():
    evaluations = all_evaluations[model_name]
    detection_rates = []
    
    for cov_thresh in coverage_thresholds:
        detected = sum(1 for e in evaluations 
                       if e.coverage >= cov_thresh and e.separation >= SEPARATION_THRESHOLD)
        detection_rates.append(detected / len(evaluations) * 100)
    
    axes[0].plot(coverage_thresholds, detection_rates, 'o-', 
                 color=MODEL_COLORS[model_name], label=model_name, linewidth=2, markersize=8)

axes[0].axvline(COVERAGE_THRESHOLD, color='gray', linestyle='--', alpha=0.7,
                label=f'Current ({COVERAGE_THRESHOLD})')
axes[0].set_xlabel('Coverage Threshold', fontsize=12)
axes[0].set_ylabel('Detection Rate (%)', fontsize=12)
axes[0].set_title(f'Effect of Coverage Threshold\n(Separation fixed at {SEPARATION_THRESHOLD})', 
                  fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)
axes[0].set_ylim(0, 100)

# Effect of separation threshold (fixed coverage)
for model_name in PRED_PATHS.keys():
    evaluations = all_evaluations[model_name]
    detection_rates = []
    
    for sep_thresh in separation_thresholds:
        detected = sum(1 for e in evaluations 
                       if e.coverage >= COVERAGE_THRESHOLD and e.separation >= sep_thresh)
        detection_rates.append(detected / len(evaluations) * 100)
    
    axes[1].plot(separation_thresholds, detection_rates, 'o-',
                 color=MODEL_COLORS[model_name], label=model_name, linewidth=2, markersize=8)

axes[1].axvline(SEPARATION_THRESHOLD, color='gray', linestyle='--', alpha=0.7,
                label=f'Current ({SEPARATION_THRESHOLD})')
axes[1].set_xlabel('Separation Threshold', fontsize=12)
axes[1].set_ylabel('Detection Rate (%)', fontsize=12)
axes[1].set_title(f'Effect of Separation Threshold\n(Coverage fixed at {COVERAGE_THRESHOLD})',
                  fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)
axes[1].set_ylim(0, 100)

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, '07_threshold_sensitivity.png'), dpi=200, bbox_inches='tight')
plt.show()

---
## Cell 18: Save Results

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save summary
summary_path = os.path.join(OUTPUT_DIR, 'stone_detection_summary.csv')
summary_df.to_csv(summary_path, index=False)
print(f"✓ Summary saved: {summary_path}")

# Save per-stone details for each model
for model_name in PRED_PATHS.keys():
    evaluations = all_evaluations[model_name]
    
    stone_data = []
    for e in evaluations:
        stone_data.append({
            'stone_id': e.gt_stone.stone_id,
            'gt_class': e.gt_stone.class_name,
            'pixel_count': e.gt_stone.pixel_count,
            'coverage': e.coverage,
            'wrong_class_coverage': e.wrong_class_coverage,
            'separation': e.separation,
            'status': e.status,
            'num_predictions': e.num_predictions_involved
        })
    
    stone_df = pd.DataFrame(stone_data)
    stone_path = os.path.join(OUTPUT_DIR, f'stone_details_{model_name.replace("-", "_")}.csv')
    stone_df.to_csv(stone_path, index=False)
    print(f"✓ Stone details saved: {stone_path}")

print("\n✓ All results saved.")

---
## Cell 19: Summary for Paper

In [ ]:
print("="*80)
print("SUMMARY FOR PAPER")
print("="*80)

print(f"\nDataset: {len(gt_stones)} individual stones")
print(f"Evaluation: Stone-level with coverage and separation metrics")
print(f"Coverage threshold: {COVERAGE_THRESHOLD} | Separation threshold: {SEPARATION_THRESHOLD}")

print("\nPer-class stone counts:")
for class_name in ['Ashlar', 'Polygonal', 'Quarry Stone']:
    count = sum(1 for s in gt_stones if s.class_name == class_name)
    print(f"  {class_name}: {count}")

print("\n" + "-"*80)
print("Model Comparison:")
print("-"*80)

for model_name in PRED_PATHS.keys():
    metrics = all_metrics[model_name]
    print(f"\n{model_name}:")
    print(f"  Detection Rate: {metrics['detection_rate']*100:.1f}%")
    print(f"  Merge Rate: {metrics['merge_rate']*100:.1f}%")
    print(f"  Mean Coverage: {metrics['mean_coverage']*100:.1f}%")
    print(f"  Mean Separation: {metrics['mean_separation']*100:.1f}%")

print("\n" + "="*80)